In [1]:
import requests
import json
import os
from dotenv import load_dotenv
load_dotenv()
import time
import langextract as lx
import textwrap
from neo4j import GraphDatabase
import chromadb
from chromadb.utils import embedding_functions

In [11]:
API_KEY = os.getenv("TMDB_API_KEY")
BASE_URL = "https://api.themoviedb.org/3"
OUT_DIR = "tmdb_raw_data"

os.makedirs(OUT_DIR, exist_ok=True)

def get(url, params=None):
    params = params or {}
    params["api_key"] = API_KEY
    r = requests.get(url, params=params)
    if r.status_code != 200:
        return None
    return r.json()

def fetch_movies(pages=1):
    movies = []
    for p in range(1, pages + 1):
        data = get(f"{BASE_URL}/movie/popular", {"page": p})
        if data:
            movies.extend(data["results"])
        time.sleep(0.2)
    return movies

def fetch_full(movie_id):
    movie = get(f"{BASE_URL}/movie/{movie_id}")
    time.sleep(0.1)
    credits = get(f"{BASE_URL}/movie/{movie_id}/credits")
    time.sleep(0.1)
    keywords = get(f"{BASE_URL}/movie/{movie_id}/keywords")
    time.sleep(0.1)
    return {"movie": movie, "credits": credits, "keywords": keywords}

movies = fetch_movies(pages=2)
dataset = []

for m in movies:
    full = fetch_full(m["id"])
    if full["movie"]:
        dataset.append(full)
        with open(f"{OUT_DIR}/{m['id']}.json", "w", encoding="utf-8") as f:
            json.dump(full, f, indent=2)

with open(f"{OUT_DIR}/all.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

print("DONE", len(dataset), "movies saved")

DONE 40 movies saved


In [12]:
print(f"{len(results)} / {len(movies)} done")

2 / 40 done


In [23]:
clean_graph = {}

for item in results:
    movie = item["movie"]

    directors = set()
    actors = set()
    genres = set()
    keywords = set()
    edges = set()

    for e in item["extractions"]:
        t = e.extraction_text
        c = e.extraction_class
        a = e.attributes or {}

        if c == "director":
            directors.add(t)
        elif c == "actor":
            actors.add(t)
        elif c == "genre":
            genres.add(t)
        elif c == "keyword":
            keywords.add(t)
        elif c == "relationship":
            frm = a.get("from")
            to = a.get("to")
            typ = a.get("type", "RELATED_TO")
            if frm and to:
                edges.add((frm, to, typ))

    clean_graph[movie] = {
        "directors": directors,
        "actors": actors,
        "genres": genres,
        "keywords": keywords,
        "edges": edges
    }

print("CLEAN DONE")

KeyError: 'movie'

In [15]:
URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "passpass")

driver = GraphDatabase.driver(URI, auth=AUTH)



def insert(tx, movie, data):
    tx.run("MERGE (m:Movie {name:$m})", m=movie)

    for d in data["directors"]:
        tx.run("""
            MERGE (p:Person {name:$p})
            SET p:Director
            WITH p MATCH (m:Movie {name:$m})
            MERGE (p)-[:DIRECTED]->(m)
        """, p=d, m=movie)

    for a in data["actors"]:
        tx.run("""
            MERGE (p:Person {name:$p})
            SET p:Actor
            WITH p MATCH (m:Movie {name:$m})
            MERGE (p)-[:ACTED_IN]->(m)
        """, p=a, m=movie)

    for g in data["genres"]:
        tx.run("""
            MERGE (g:Genre {name:$g})
            WITH g MATCH (m:Movie {name:$m})
            MERGE (m)-[:HAS_GENRE]->(g)
        """, g=g, m=movie)

    for k in data["keywords"]:
        tx.run("""
            MERGE (k:Keyword {name:$k})
            WITH k MATCH (m:Movie {name:$m})
            MERGE (m)-[:HAS_KEYWORD]->(k)
        """, k=k, m=movie)

    for frm, to, typ in data["edges"]:
        allowed = {"DIRECTED", "ACTED_IN", "HAS_GENRE", "HAS_KEYWORD"}
        if typ not in allowed:
            typ = "RELATED_TO"
        tx.run(f"""
            MERGE (a:Person {{name:$a}})
            MERGE (b {{name:$b}})
            MERGE (a)-[:{typ}]->(b)
        """, a=frm, b=to)

with driver.session() as session:
    for movie, data in clean_graph.items():
        session.execute_write(insert, movie, data)

driver.close()
print("NEO4J DONE")

NEO4J DONE


In [16]:

NEO4J_URI = "bolt://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "passpass"  

MODEL_URL = "http://localhost:11434"
MODEL_ID = "qwen2.5:7b" 

def run_cypher(query):
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    try:
        with driver.session() as session:
            result = session.run(query)
            return [record.data() for record in result]
    finally:
        driver.close()

def llm(prompt):
    response = requests.post(
        f"{MODEL_URL}/api/generate",
        json={
            "model": MODEL_ID,
            "prompt": prompt,
            "stream": False
        }
    )
    return response.json().get("response", "No response").strip()

def neo4j_rag(question):
    print(f"\nQuestion: {question}")
    print("-" * 50)

    intent = llm(f"""Given this question about movies, extract:
1. The intent (e.g. find movies, find directors, count, recommend)
2. Any specific names mentioned (movies, people, genres, keywords)

Reply in this exact format:
INTENT: <intent>
ENTITIES: <comma separated names or NONE>

Question: {question}""")
    
    print(f"Intent parsed:\n{intent}\n")

    cypher = llm(f"""You are a Neo4j expert. Write a Cypher query for this question.

SCHEMA:
- (Person)-[:DIRECTED]->(Movie)
- (Person)-[:ACTED_IN]->(Movie)  
- (Movie)-[:HAS_GENRE]->(Genre)
- (Movie)-[:HAS_KEYWORD]->(Keyword)
- All names are case-sensitive and title-cased e.g. 'Action' not 'action'

CRITICAL CYPHER RULES:
- Cypher does NOT have GROUP BY or HAVING — never use them.
- To group and filter counts use WITH: MATCH ... WITH n, count(m) AS c WHERE c > 1 RETURN n.name, c
- To count relationships use WITH before filtering.

EXAMPLES:
Q: Which directors directed more than one movie?
A: MATCH (p:Person)-[:DIRECTED]->(m:Movie) WITH p, count(m) AS c WHERE c > 1 RETURN p.name, c ORDER BY c DESC LIMIT 10

Q: Which movie has the most keywords?
A: MATCH (m:Movie)-[:HAS_KEYWORD]->(k:Keyword) WITH m, count(k) AS c RETURN m.name, c ORDER BY c DESC LIMIT 10

Q: What genres does Sam Raimi work in?
A: MATCH (p:Person {{name: 'Sam Raimi'}})-[:DIRECTED]->(m:Movie)-[:HAS_GENRE]->(g:Genre) RETURN g.name LIMIT 10

INTENT ANALYSIS:
{intent}

QUESTION: {question}

Return ONLY the Cypher query, no markdown, no backticks, no explanation.
Cypher:""")

    print(f"Cypher:\n{cypher}\n")

    try:
        records = run_cypher(cypher)
    except Exception as e:
        print(f"Error: {e} — asking LLM to fix...")
        cypher = llm(f"""This Cypher failed: {cypher}
Error: {e}

REMEMBER: Cypher has no GROUP BY or HAVING. Use WITH to filter aggregations:
MATCH (p:Person)-[:DIRECTED]->(m:Movie) WITH p, count(m) AS c WHERE c > 1 RETURN p.name, c

Return ONLY the fixed Cypher.""")
        print(f"Fixed Cypher:\n{cypher}\n")
        try:
            records = run_cypher(cypher)
        except Exception as e2:
            print(f"Failed again: {e2}")
            return

    print(f"Raw results: {records}\n")

    answer = llm(f"""You are a movie expert. Answer this question using the graph database results below.

QUESTION: {question}

GRAPH RESULTS:
{records}

Give a clear, direct answer based only on these results. If empty, say the information was not found in the graph.""")

    print(f"Answer: {answer}")
    return {"cypher": cypher, "records": records, "answer": answer}


print("Neo4j:", run_cypher("RETURN 1 AS test"))
print("LLM:", llm("Say hello in one word."))

Neo4j: [{'test': 1}]
LLM: Hi!


In [17]:
MOVIES_FOLDER = r"C:\Users\omarl\Desktop\LLM_RAG\tmdb_raw_data"
CHROMA_PATH = "chroma_db"


client = chromadb.PersistentClient(path=CHROMA_PATH)

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="movies",
    embedding_function=ef
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4908.52it/s]


In [18]:

def chunk_text(text, size=3000):
    return [text[i:i+size] for i in range(0, len(text), size)]


files = [f for f in os.listdir(MOVIES_FOLDER) if f.endswith(".json")]

for fname in files:
    fpath = os.path.join(MOVIES_FOLDER, fname)

    with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError as e:
            print(f"Skipping {fname}: {e}")
            continue

    text = json.dumps(data)
    chunks = chunk_text(text)
    movie_name = fname.replace(".json", "")

    for i, chunk in enumerate(chunks):
        collection.upsert(
            ids=[f"{movie_name}_{i}"],
            documents=[chunk],
            metadatas=[{"movie": movie_name, "chunk": i}]
        )

    print(f"Added {fname} -> {len(chunks)} chunks")

print(f"\nDone — {collection.count()} chunks indexed from {len(files)} files")

Added 1007757.json -> 10 chunks
Added 1065834.json -> 6 chunks
Added 1083381.json -> 12 chunks
Added 11012.json -> 6 chunks
Added 1116201.json -> 19 chunks
Added 1226863.json -> 9 chunks
Added 1228710.json -> 16 chunks
Added 1252051.json -> 4 chunks
Added 1292415.json -> 5 chunks
Added 1297842.json -> 32 chunks
Added 1304313.json -> 9 chunks
Added 1311031.json -> 40 chunks
Added 1318447.json -> 9 chunks
Added 1320660.json -> 6 chunks
Added 1327819.json -> 14 chunks
Added 1339713.json -> 13 chunks
Added 1367220.json -> 10 chunks
Added 1371023.json -> 5 chunks
Added 1380291.json -> 8 chunks
Added 1390300.json -> 6 chunks
Added 1419406.json -> 8 chunks
Added 1426822.json -> 2 chunks
Added 1431071.json -> 5 chunks
Added 1433117.json -> 7 chunks
Added 1439930.json -> 42 chunks
Added 1472951.json -> 5 chunks
Added 1510339.json -> 6 chunks
Added 1523145.json -> 6 chunks
Added 1582770.json -> 11 chunks
Added 249397.json -> 11 chunks
Added 350.json -> 17 chunks
Added 405818.json -> 4 chunks
Add

In [1]:
MODEL_URL = "http://localhost:11434"
MODEL_ID = "qwen2.5:7b"  
def rag_search(question, n=5):
    print(f"\nQuestion: {question}")
    print("-" * 50)

    
    intent = llm(f"""Given this question about movies, extract:
1. The intent (e.g. find movies, find directors, count, recommend)
2. Any specific names mentioned (movies, people, genres, keywords)
Reply in this exact format:
INTENT: <intent>
ENTITIES: <comma separated names or NONE>
Question: {question}""")

    print(f"Intent parsed:\n{intent}\n")

    
    search_query = llm(f"""You are a search query optimizer for a movie vector database.
Given the intent and question, generate the best possible search query to retrieve relevant movies.
- Be specific and include genre, themes, character types, mood, or keywords
- Do NOT include filler words
- Return ONLY the search query string, nothing else
INTENT ANALYSIS:
{intent}
QUESTION: {question}
Search query:""")

    print(f"Optimized search query: {search_query}\n")

    
    try:
        res = collection.query(query_texts=[search_query.strip()], n_results=n)
        retrieved = res["documents"][0]
        metadatas = res["metadatas"][0]
        titles = [m.get("movie", m.get("title", "unknown")) for m in metadatas]
    except Exception as e:
        print(f"Chroma query failed: {e}")
        return

    print(f"Retrieved: {titles}\n")

    context = "\n\n".join(
        f"Movie: {t}\n{d}" for t, d in zip(titles, retrieved)
    )

    answer = llm(f"""You are a movie expert. Based on the following movies retrieved from a database, answer the user's question.
MOVIES:
{context}

QUESTION: {question}

Give a clear, direct answer based only on the movies provided. If the information is not found, say so.""")

    print(f"Answer: {answer}")

    return {"search_query": search_query, "titles": titles, "answer": answer}

In [2]:
questions = [
    
    {"id": 1, "question": "Recommend me something to feel good and lighthearted", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 2, "question": "Find me movies with a similar vibe to Malena", "category": "semantic_similarity", "expected_winner": "chroma"},
    {"id": 3, "question": "What should I watch if I want something emotional but not too depressing?", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 4, "question": "Suggest a movie that feels nostalgic and romantic", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 5, "question": "I want a slow, atmospheric movie with beautiful visuals", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 6, "question": "Recommend something tense and psychological", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 7, "question": "Find movies that feel dreamy and poetic", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 8, "question": "What movies are good for a cozy weekend night?", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 9, "question": "Recommend something dark, stylish, and suspenseful", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 10, "question": "Find a movie with a bittersweet coming-of-age feeling", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 11, "question": "Suggest films about complicated love and memory", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 12, "question": "I want something whimsical but emotionally meaningful", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 13, "question": "Find movies that are quiet, intimate, and character-driven", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 14, "question": "Recommend a movie about loneliness in a big city", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 15, "question": "What should I watch if I liked the emotional tone of Lost in Translation?", "category": "semantic_similarity", "expected_winner": "chroma"},
    {"id": 16, "question": "Suggest something eerie without being a straightforward horror movie", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 17, "question": "Find films with themes of obsession and identity", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 18, "question": "Recommend movies that feel warm, funny, and human", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 19, "question": "I want a stylish crime movie with a cool atmosphere", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 20, "question": "Find something sad, beautiful, and reflective", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 21, "question": "Recommend a movie that explores grief in a subtle way", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 22, "question": "What movies have a magical, fairy-tale-like mood?", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 23, "question": "Find films similar in mood to Amelie", "category": "semantic_similarity", "expected_winner": "chroma"},
    {"id": 24, "question": "Recommend something philosophical but still entertaining", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 25, "question": "I want a movie about second chances and redemption", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 26, "question": "Suggest a film with a strong sense of place and atmosphere", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 27, "question": "Find movies that are emotionally intense and visually striking", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 28, "question": "Recommend something clever, playful, and unusual", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 29, "question": "What should I watch if I want a melancholic romance?", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 30, "question": "Find movies about friendship that are not cheesy", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 31, "question": "Suggest a movie with an unsettling small-town atmosphere", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 32, "question": "Recommend a film that feels like a memory or dream", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 33, "question": "Find something adventurous but emotionally grounded", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 34, "question": "What movies have a haunting and tragic love story?", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 35, "question": "Suggest a movie with quiet humor and gentle drama", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 36, "question": "Find a movie that captures youthful confusion and longing", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 37, "question": "Recommend something with moral ambiguity and complex characters", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 38, "question": "I want a movie that feels elegant, romantic, and tragic", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 39, "question": "Find films with a surreal and mysterious atmosphere", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 40, "question": "Recommend something uplifting without being silly", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 41, "question": "Suggest a movie about family secrets and emotional tension", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 42, "question": "Find a film that feels raw, realistic, and human", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 43, "question": "What should I watch if I like tragic historical romances?", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 44, "question": "Recommend something fast-paced and fun but not dumb", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 45, "question": "Find movies with a lonely outsider protagonist", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 46, "question": "Suggest something that mixes humor with sadness", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 47, "question": "Recommend a movie with a moody noir feeling", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 48, "question": "Find movies that are romantic but unconventional", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 49, "question": "I want a reflective film about aging and regret", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 50, "question": "Suggest films that feel emotionally honest", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 51, "question": "Recommend something with dark comedy and social satire", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 52, "question": "Find movies similar in feeling to Cinema Paradiso", "category": "semantic_similarity", "expected_winner": "chroma"},
    {"id": 53, "question": "What should I watch for a romantic European art-house mood?", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 54, "question": "Suggest a film that is mysterious, emotional, and slow-burning", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 55, "question": "Find movies about forbidden love", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 56, "question": "Recommend something that feels grand, epic, and emotional", "category": "semantic_vibe", "expected_winner": "chroma"},
    {"id": 57, "question": "I want a movie with a charming but flawed main character", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 58, "question": "Find movies with a hopeful ending after hardship", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 59, "question": "Suggest a film that explores jealousy and desire", "category": "semantic_theme", "expected_winner": "chroma"},
    {"id": 60, "question": "Recommend a movie that feels tender and sad", "category": "semantic_vibe", "expected_winner": "chroma"},

    # Expected Neo4j wins
    {"id": 61, "question": "Which movies have Action as a genre?", "category": "structured_genre_lookup", "expected_winner": "neo4j"},
    {"id": 62, "question": "What genres does Sam Raimi work in?", "category": "graph_traversal", "expected_winner": "neo4j"},
    {"id": 63, "question": "Which movie has the most keywords tagged to it?", "category": "aggregation", "expected_winner": "neo4j"},
    {"id": 64, "question": "List movies in the Comedy genre", "category": "structured_genre_lookup", "expected_winner": "neo4j"},
    {"id": 65, "question": "Which people directed movies in the Horror genre?", "category": "graph_traversal", "expected_winner": "neo4j"},
    {"id": 66, "question": "Which movies are connected to the keyword romance?", "category": "structured_keyword_lookup", "expected_winner": "neo4j"},
    {"id": 67, "question": "What actors appeared in Spider-Man?", "category": "structured_cast_lookup", "expected_winner": "neo4j"},
    {"id": 68, "question": "Who directed Spider-Man?", "category": "structured_director_lookup", "expected_winner": "neo4j"},
    {"id": 69, "question": "Which movies did Kirsten Dunst act in?", "category": "structured_actor_lookup", "expected_winner": "neo4j"},
    {"id": 70, "question": "Which genres are attached to Malena?", "category": "structured_movie_lookup", "expected_winner": "neo4j"},
    {"id": 71, "question": "Which movies share a genre with Malena?", "category": "graph_traversal", "expected_winner": "neo4j"},
    {"id": 72, "question": "Which movies share keywords with Malena?", "category": "graph_traversal", "expected_winner": "neo4j"},
    {"id": 73, "question": "Which genre has the most movies?", "category": "aggregation", "expected_winner": "neo4j"},
    {"id": 74, "question": "Which actor appears in the most movies?", "category": "aggregation", "expected_winner": "neo4j"},
    {"id": 75, "question": "Which director has directed the most movies?", "category": "aggregation", "expected_winner": "neo4j"},
    {"id": 76, "question": "Show movies that are both Drama and Romance", "category": "structured_multi_filter", "expected_winner": "neo4j"},
    {"id": 77, "question": "Show movies that are both Action and Adventure", "category": "structured_multi_filter", "expected_winner": "neo4j"},
    {"id": 78, "question": "Which people worked on movies tagged with friendship?", "category": "graph_traversal", "expected_winner": "neo4j"},
    {"id": 79, "question": "Which movies are tagged with both love and jealousy?", "category": "structured_multi_filter", "expected_winner": "neo4j"},
    {"id": 80, "question": "What keywords are connected to The Godfather?", "category": "structured_movie_lookup", "expected_winner": "neo4j"},
    {"id": 81, "question": "What genres are connected to The Godfather?", "category": "structured_movie_lookup", "expected_winner": "neo4j"},
    {"id": 82, "question": "Which movies did Francis Ford Coppola direct?", "category": "structured_director_lookup", "expected_winner": "neo4j"},
    {"id": 83, "question": "Which actors worked with Francis Ford Coppola?", "category": "graph_traversal", "expected_winner": "neo4j"},
    {"id": 84, "question": "Which directors have worked with Al Pacino?", "category": "graph_traversal", "expected_winner": "neo4j"},
    {"id": 85, "question": "Which movies include Al Pacino?", "category": "structured_actor_lookup", "expected_winner": "neo4j"},
    {"id": 86, "question": "Count how many movies are in each genre", "category": "aggregation", "expected_winner": "neo4j"},
    {"id": 87, "question": "Count how many keywords each movie has", "category": "aggregation", "expected_winner": "neo4j"},
    {"id": 88, "question": "Which keyword is used by the most movies?", "category": "aggregation", "expected_winner": "neo4j"},
    {"id": 89, "question": "Find actors who appeared in more than one movie", "category": "aggregation", "expected_winner": "neo4j"},
    {"id": 90, "question": "Find directors who directed movies in more than one genre", "category": "aggregation", "expected_winner": "neo4j"},
    {"id": 91, "question": "Which movies are connected to the keyword mafia?", "category": "structured_keyword_lookup", "expected_winner": "neo4j"},
    {"id": 92, "question": "Which genres does Al Pacino appear in?", "category": "graph_traversal", "expected_winner": "neo4j"},
    {"id": 93, "question": "Which people are connected to both Crime and Drama movies?", "category": "graph_traversal", "expected_winner": "neo4j"},
    {"id": 94, "question": "Which movies have exactly the genre Drama?", "category": "structured_genre_lookup", "expected_winner": "neo4j"},
    {"id": 95, "question": "Which movies have the fewest keywords?", "category": "aggregation", "expected_winner": "neo4j"},
    {"id": 96, "question": "Which actors co-starred with Tobey Maguire?", "category": "graph_traversal", "expected_winner": "neo4j"},
    {"id": 97, "question": "Which directors made movies tagged with superhero?", "category": "graph_traversal", "expected_winner": "neo4j"},
    {"id": 98, "question": "Which movies are tagged with superhero?", "category": "structured_keyword_lookup", "expected_winner": "neo4j"},
    {"id": 99, "question": "Which genres are most common among movies starring Tobey Maguire?", "category": "aggregation", "expected_winner": "neo4j"},
    {"id": 100, "question": "Which movie has the largest number of connected people?", "category": "aggregation", "expected_winner": "neo4j"},
]


print("=" * 60)
print("CHROMA RAG vs NEO4J RAG COMPARISON")
print("=" * 60)

chroma_wins = 0
neo4j_wins = 0

results = []

for q in questions:
    print(f"\nQ{q['id']}: {q['question']}")
    print("-" * 60)


    start = time.time()
    chroma_result = rag_search(q["question"])
    chroma_time = time.time() - start

    start = time.time()
    neo4j_result = neo4j_rag(q["question"])
    neo4j_time = time.time() - start

    print(f"Chroma Time : {chroma_time:.2f}s")
    print(f"Neo4j Time  : {neo4j_time:.2f}s")


    if chroma_time < neo4j_time:
        speed_winner = "chroma"
        chroma_wins += 1
    else:
        speed_winner = "neo4j"
        neo4j_wins += 1

    expected = q["expected_winner"]

    results.append({
        "id": q["id"],
        "expected": expected,
        "speed_winner": speed_winner,
        "chroma_time": round(chroma_time, 2),
        "neo4j_time": round(neo4j_time, 2)
    })

    print(f"Expected Winner : {expected}")
    print(f"Faster System   : {speed_winner}")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Chroma faster in {chroma_wins} questions")
print(f"Neo4j faster in {neo4j_wins} questions")

avg_chroma = sum(r["chroma_time"] for r in results) / len(results)
avg_neo4j = sum(r["neo4j_time"] for r in results) / len(results)

print(f"\nAverage Chroma Time: {avg_chroma:.2f}s")
print(f"Average Neo4j Time : {avg_neo4j:.2f}s")

CHROMA RAG vs NEO4J RAG COMPARISON

Q1: Recommend me something to feel good and lighthearted
------------------------------------------------------------


NameError: name 'time' is not defined

In [ ]:
import pandas as pd


total_questions = len(questions)

semantic_questions = [q for q in questions if q["expected_winner"] == "chroma"]
structured_questions = [q for q in questions if q["expected_winner"] == "neo4j"]

semantic_total = len(semantic_questions)
structured_total = len(structured_questions)

semantic_correct = sum(
    1
    for r, q in zip(results, questions)
    if q["expected_winner"] == "chroma"
    and r["speed_winner"] == "chroma"
)

structured_correct = sum(
    1
    for r, q in zip(results, questions)
    if q["expected_winner"] == "neo4j"
    and r["speed_winner"] == "neo4j"
)

overall_correct = semantic_correct + structured_correct

avg_chroma_time = sum(r["chroma_time"] for r in results) / total_questions
avg_neo4j_time = sum(r["neo4j_time"] for r in results) / total_questions

summary = pd.DataFrame({
    "Metric": [
        "Semantic Questions Accuracy",
        "Structured Questions Accuracy",
        "Overall Routing Accuracy",
        "Average Chroma Response Time (s)",
        "Average Neo4j Response Time (s)"
    ],
    "Value": [
        f"{semantic_correct}/{semantic_total} ({semantic_correct/semantic_total*100:.1f}%)",
        f"{structured_correct}/{structured_total} ({structured_correct/structured_total*100:.1f}%)",
        f"{overall_correct}/{total_questions} ({overall_correct/total_questions*100:.1f}%)",
        f"{avg_chroma_time:.2f}",
        f"{avg_neo4j_time:.2f}"
    ]
})


print(summary.to_string(index=False))

                          Metric          Value
     Semantic Questions Accuracy   6/60 (10.0%)
   Structured Questions Accuracy  39/40 (97.5%)
        Overall Routing Accuracy 45/100 (45.0%)
Average Chroma Response Time (s)          13.33
 Average Neo4j Response Time (s)          11.00


In [ ]:
#On remarque que Neo4j est beaucoup plus performant sur les questions structurées especially celles liées aux relations entre acteurs films et genres.
#Chroma works better sur les questions sémantiques mais les résultats sont moins stables que prévu dans certains cas.
#Le Rag favorise Neo4j ce qui montre que le routing est peut-être biaised vers les requêtes structurées.
#Les temps de réponse sont assez proches mais Neo4j reste légèrement plus rapide en moyenne.